# delhi-weather: forecasting (batch, databricks-aml-step)

**Use case:** `usecase-type/batch/timeseries-forecasting/databricks-azureml-pipeline-step` — bounded scoring of many requests into stored predictions with the Azure ML pipeline running Databricks steps.

**Training:** Azure ML pipeline whose command steps run the existing Databricks job tasks with managed identity and pass the evaluated MLflow model URI between steps.

**Serving:** The second pipeline step runs the Databricks `batch-score` task.

**Status:** Executable notebook templates over shared engines; cloud steps are explicit, charged operations behind switches that default to false and need the listed prerequisites.

Kaggle-only source: `sumanthvrao/daily-climate-time-series-data`.
Dataset status: **configured**.
Daily mean temperature; weather covariates are omitted because their future values are not known at forecast origin. Last two 14-day windows of the labeled training file are held out. Verify upstream Weather Underground attribution for intended use.

**Prerequisites**

- Review the scenario's dataset status, license and notes; gated datasets stay blocked until approved.
- For Azure steps: `user-config/runtime.local.json` naming an existing workspace, compute, identity and storage selection.
- An existing Databricks workspace and cluster with the shared notebooks and factory wheel imported; Unity Catalog objects and permissions per `user-config/databricks`.
- An Azure ML compute identity allowed to run the existing Databricks job (no PAT tokens).

**Limitations**

- The custom seasonal-naive baseline only forecasts timestamps inside its fitted horizon window.
- Databricks models live in Databricks MLflow/Unity Catalog, not in the Azure ML registry.
- Templates are not evidence of a completed Kaggle, Azure or Databricks run.

This notebook is an executable template, not evidence of a completed Kaggle or Azure run.
Install the factory's declared extras into this kernel separately. No packages,
credentials, cloud resources, or competition terms are installed/accepted by this notebook.
All execution switches default to false. Run **either** SDK v2 **or** CLI v2 submission, not both.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "pyproject.toml").is_file()
             and (p / "accelerator" / "src" / "ml_model_factory" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the 50-ml-model-factory checkout")
SOURCE = ROOT / "accelerator" / "src"
sys.path.insert(0, str(SOURCE))
from ml_model_factory.config import load_json, validate_scenario, validate_runtime, write_json

SCENARIO = ROOT / "user-config" / "model" / "scenarios" / "delhi-weather.json"
scenario = validate_scenario(load_json(SCENARIO))
MODE = "custom"
PATTERN = "batch"
TECHNOLOGY = "databricks-azureml-pipeline-step"
RUNTIME = Path("user-config/runtime.local.json")
if not RUNTIME.is_absolute():
    RUNTIME = ROOT / RUNTIME
WORK = ROOT / "ml-environment" / "outputs" / scenario["name"] / "databricks-aml-step"
RAW = ROOT / "data" / "in" / scenario["name"]
PREPARED = ROOT / "data" / "out" / scenario["name"] / "databricks-aml-step" / "prepared"
MODEL = WORK / "model"
EVALUATION = WORK / "evaluation"
RENDERED = WORK / "azureml"
INFERENCE = WORK / PATTERN
REQUESTS = INFERENCE / "requests"
MODEL_NAME = scenario.get("model_name", scenario["name"])
MODEL_ID = ""  # immutable azureml:<name>:<version> after quality-gated registration
RENDER_DATABRICKS_JOB = False
CREATE_DATABRICKS_JOB = False
RENDER_DATABRICKS_PIPELINE = False
SUBMIT_SDK = False
SUBMIT_CLI = False
DATABRICKS_HOST = ""  # https://adb-<workspace-id>.<n>.azuredatabricks.net
DATABRICKS_SCENARIO_PATH = "/Workspace/Shared/ml-model-factory/user-config/model/scenarios/" + scenario["name"] + ".json"
LAKE_CONFIG = ROOT / "user-config" / "lake.local.json"
LAKE_ROOT = ROOT / "data" / "out" / "lake-data"
EFFECTIVE_RUNTIME = RUNTIME
FACTORY_ENV = os.environ.copy()
FACTORY_ENV["PYTHONPATH"] = str(SOURCE) + os.pathsep + FACTORY_ENV.get("PYTHONPATH", "")
FACTORY_ENV.setdefault("MLFLOW_TRACKING_URI", (ROOT / "ml-environment" / "mlruns").as_uri())

def factory(*arguments):
    command = [sys.executable, "-m", "ml_model_factory", *map(str, arguments)]
    subprocess.run(command, cwd=ROOT, env=FACTORY_ENV, check=True)


## Databricks job for Azure ML orchestration

The Azure ML steps run single tasks of an existing Databricks job (`only` task runs), so the
job is rendered with `--orchestration azureml`: `batch-score` reads the model URI from a job
parameter that the second Azure ML step supplies. Create it once, then record its job ID in
`user-config/databricks/azureml-step.local.json`.


In [ ]:
DATABRICKS_TEMPLATE = ROOT / "user-config" / "databricks" / "job-template.json"
DATABRICKS_SETTINGS = ROOT / "user-config" / "databricks" / "inference-settings.local.json"
DATABRICKS_JOB = INFERENCE / "databricks-job.json"
if RENDER_DATABRICKS_JOB:
    factory("databricks-job", "render", "--scenario", SCENARIO, "--pattern", PATTERN,
            "--template", DATABRICKS_TEMPLATE, "--settings", DATABRICKS_SETTINGS,
            "--scenario-path", DATABRICKS_SCENARIO_PATH, "--orchestration", "azureml",
            "--output", DATABRICKS_JOB)
if CREATE_DATABRICKS_JOB:
    factory("databricks-job", "create", "--job", DATABRICKS_JOB, "--host", DATABRICKS_HOST,
            "--auth", "azure-cli", "--execute")


## Azure ML pipeline with Databricks steps

`databricks-pipeline render` writes an Azure ML v2 pipeline whose first command step runs only the
existing Databricks job's `train-evaluate` task and whose second step, bound to the first step's
output, runs only `batch-score` with the returned model URI. Steps authenticate with the compute's
managed identity and have explicit timeouts. Configure the job ID, task keys and workspace in
`user-config/databricks/azureml-step.local.json` (see the example file).


In [ ]:
DATABRICKS_STEP_CONFIG = ROOT / "user-config" / "databricks" / "azureml-step.local.json"
if RENDER_DATABRICKS_PIPELINE:
    factory("databricks-pipeline", "render", "--scenario", SCENARIO, "--runtime", EFFECTIVE_RUNTIME,
            "--pattern", PATTERN, "--databricks", DATABRICKS_STEP_CONFIG, "--output", RENDERED)
JOB_PATH = RENDERED / "pipeline.yml"


## Azure ML SDK v2 submission — alternative A


In [ ]:
if SUBMIT_SDK:
    if SUBMIT_CLI:
        raise ValueError("Choose one submission route to avoid duplicate charged jobs")
    from ml_model_factory.azureml import submit
    runtime = validate_runtime(load_json(EFFECTIVE_RUNTIME))
    if not JOB_PATH.exists():
        raise RuntimeError("Render and review the job before submission")
    # Shared SDK v2 submission binds the configured tenant and rejects unresolved inputs.
    print(submit(JOB_PATH, runtime))


## Azure ML CLI v2 submission — alternative B


In [ ]:
if SUBMIT_CLI:
    if SUBMIT_SDK:
        raise ValueError("Choose one submission route to avoid duplicate charged jobs")
    runtime = validate_runtime(load_json(EFFECTIVE_RUNTIME))
    if not JOB_PATH.exists():
        raise RuntimeError("Render and review the job before submission")
    subprocess.run([sys.executable, str(ROOT / "accelerator" / "scripts" / "azureml_cli.py"),
                    "--runtime", str(EFFECTIVE_RUNTIME), "--job", str(JOB_PATH)],
                   cwd=ROOT, env=FACTORY_ENV, check=True)

# Equivalent factory SDK-backed submission:
# factory("submit", "--runtime", RUNTIME, "--job", JOB_PATH)


## Monitoring and clean-up

Join predictions with observed labels by `request_id` only after outcomes are known, then use
`python -m ml_model_factory monitor` with explicit reference/current windows (see the
model-factory guide). Predictions are never labels. Batch compute scales back to zero; storage of inputs and predictions remains.
